#  ⭐ `dim_via_administracao`


In [1]:
%run ../../config/bootstrap.py


from pathlib import Path
from utils import get_project_root , normalizar_via_administracao#, normalizar_via_fuzzy

import re
import pandas as pd
from unidecode import unidecode

In [2]:
project_root = get_project_root() 

# 🥉 Bronze 

Raw Data
as is production, low quality

In [3]:
path = Path(project_root) / "data/01_bronze/Medicamentos/Medicamentos.parquet"
bronze = pd.read_parquet(Path(project_root) / "data/01_bronze/Medicamentos/Medicamentos.parquet") 
bronze =  bronze[["VIA_ADMINISTRACAO_MAE_PAI","VIA_ADMINISTRACAO"]].drop_duplicates()
bronze.columns

Index(['VIA_ADMINISTRACAO_MAE_PAI', 'VIA_ADMINISTRACAO'], dtype='object')

In [7]:
bronze.VIA_ADMINISTRACAO.nunique()

1837

In [5]:
bronze.VIA_ADMINISTRACAO_MAE_PAI.nunique()

23

In [6]:
bronze.to_csv(Path(project_root) / "data/01_bronze/Medicamentos/Medicamentos_VIA_ADM.csv", sep=";", index=False)

# 🥈 Silver

normalized data, medium quality


In [25]:
import pandas as pd
from unidecode import unidecode
from rapidfuzz import process, fuzz

# 1. Palavras canônicas -> códigos finais
CANONICAL_KEYWORDS = {
    "Implant": [
        "implant",
        "intrauterine delivery system",
        "intrauterino",
        "intra-uterino","braço"
    ],
    "Inhal": [
        "inhalacao", "inhalation", "inalatorio", "inalatoria",
        "aerosol", "nebulizacao", "respiratoria", "uso inalatorio",
    ],
    "Instill": [
        "oftalmica", "oftalmico", "ophthalmic",
        "colirio", "ocular", "otologica", "auricular","ocular"
    ],
    "N": [
        "nasal", "intranasal", "spray nasal",
    ],
    "O": [
        "via oral", "oral", "uso oral",
        "oral use",
        "enteral", "gastroenteral", "gastroenterica",
        "sonda nasogastrica", "nasogastric tube",
        "sonda nasoenteral", "gastrostomia", "jejunostomia",
    ],
    "P": [
        "intravenosa", "intravenoso", "endovenosa", "endovenoso","ev / iv",
        "intravenous"
        "ev", "iv", "infusao", "bolus", "bomba de infusao",
        "intramuscular", "im",
        "subcutanea", "subcutaneo", "sc",
        "hipodermoclise",
        "intratecal", "intrathecal", "raquidiana", "epidural", "peridural",
        "intraperitoneal", "intracavitaria", "intravesical",
        "hemodialise", "extracorporea",
        "parenteral",
        "intravitreal", "intravitre",
        "dental",
        "transplacental", "transmamaria", "transmammary","intramuscular",
        "tranplacental"
    ],
    "R": [
        "retal", "via retal", "rectal",
    ],
    "SL": [
        "sublingual", "buccal", "bucal", "oromucosal", "oromucosa",
    ],
    "TD": [
        "topica", "topico", "tópica", "tópico",
        "cutanea", "cutaneo", "cutaneous",
        "dermatologica",
        "transdermica", "transdermico", "transdermal",
        "percutanea",
    ],
    "V": [
        "vaginal", "intravaginal",
    ],
}

# 2. Lista auxiliar para fuzzy: (palavra_canonica_normalizada, codigo)
CANONICAL_FLAT = []
for code, words in CANONICAL_KEYWORDS.items():
    for w in words:
        CANONICAL_FLAT.append(
            (unidecode(w).lower().strip(), code)
        )

UNKNOWN_TOKENS = [
    "unknown", "desconhecid", "e2b r2 code: 065", "e2b r2 code: 050",
    "nao informado", "não informado", "nao informada", "não informada",
    "nao perguntado", "não perguntado",
    "nao questionado", "não questionado",
    "via desconhecida", "rota desconhecida",
    "via de administracao nao informada",
    "via de administracao nao especificada",
    "route of administration not applicable",
]


def _preprocess(text: str) -> str:
    return " ".join(unidecode(str(text)).lower().strip().split())


def _is_unknown(s: str) -> bool:
    if s in {"", "none", "nan", "na", "n/a"}:
        return True
    return any(tok in s for tok in UNKNOWN_TOKENS)


def normalizar_via_fuzzy(value: str,
                         score_threshold: int = 80) -> str:
    """
    Normaliza via de administração usando fuzzy matching
    contra palavras canônicas definidas em CANONICAL_KEYWORDS.
    Retorna um dos códigos:
      Implant, Inhal, Instill, N, O, P, R, SL, TD, V
    ou 'desconhecido'.
    """
    if pd.isna(value):
        return "desconhecido"

    s = _preprocess(value)

    # 1) Desconhecido explícito
    if _is_unknown(s):
        return "desconhecido"

    # 2) Tratamento simples de combinações claras:
    #    se tiver oral + venoso/subcutâneo, prioriza P
    has_oral = "oral" in s
    has_parenteral_token = any(
        t in s
        for t in ("intraven", "endoven", "venos", "intramus", "subcut", "hipodermoclise")
    )
    if has_oral and has_parenteral_token:
        return "P"

    # 3) Fuzzy: compara a string s com todas as palavras canônicas
    candidates = [w for (w, _) in CANONICAL_FLAT]
    best, score, idx = process.extractOne(
        s,
        candidates,
        scorer=fuzz.partial_ratio
    )

    if score < score_threshold:
        return "desconhecido"

    # recupera o código associado à palavra canônica escolhida
    canon_word, code = CANONICAL_FLAT[idx]
    return code


In [26]:
silver = pd.read_csv(Path(project_root) / "data/01_bronze/Medicamentos/Medicamentos_VIA_ADM.csv", sep=";") ## Manual Normalization
 
silver.columns

Index(['VIA_ADMINISTRACAO_MAE_PAI', 'VIA_ADMINISTRACAO'], dtype='object')

In [14]:
silver["VIA_ADMINISTRACAO_CHAVE"] = silver["VIA_ADMINISTRACAO"].apply(normalizar_via_administracao)
silver["VIA_ADMINISTRACAO_MAE_PAI_CHAVE"] = silver["VIA_ADMINISTRACAO_MAE_PAI"].apply(normalizar_via_administracao)


In [15]:

silver["VIA_ADMINISTRACAO_MAE_PAI_CHAVE"].value_counts()


VIA_ADMINISTRACAO_MAE_PAI_CHAVE
desconhecido    1850
O                 10
P                  8
TD                 6
Implant            1
Inhal              1
Name: count, dtype: int64

In [27]:
silver["VIA_ADMINISTRACAO_CHAVE_fuzzy"] = silver["VIA_ADMINISTRACAO"].apply(normalizar_via_fuzzy)
silver["VIA_ADMINISTRACAO_MAE_PAI_CHAVE_fuzzy"] = silver["VIA_ADMINISTRACAO_MAE_PAI"].apply(normalizar_via_fuzzy)


In [29]:

silver.to_csv(Path(project_root) / "data/02_silver/hist_via_adm/MANUAL_hist_via_adm.csv", sep=";", index=False)

In [28]:
silver["VIA_ADMINISTRACAO_CHAVE_fuzzy"].value_counts()


VIA_ADMINISTRACAO_CHAVE_fuzzy
P               856
desconhecido    381
O               250
Inhal           100
TD               98
Implant          77
Instill          58
N                21
R                14
SL               13
V                 8
Name: count, dtype: int64

In [18]:
silver["VIA_ADMINISTRACAO_MAE_PAI_CHAVE_fuzzy"].value_counts()

VIA_ADMINISTRACAO_MAE_PAI_CHAVE_fuzzy
desconhecido    1848
P                 14
O                 11
Implant            2
Inhal              1
Name: count, dtype: int64

In [21]:
silver.query("VIA_ADMINISTRACAO_MAE_PAI_CHAVE_fuzzy=='desconhecido'")["VIA_ADMINISTRACAO"].drop_duplicates()

0                                                infusão intravenosa
1                                                               oral
2                                                                NaN
3                                                      intramuscular
4                                     intravenosa (não especificado)
5                                                              outra
6                                                            vaginal
7                                                         subcutânea
8                                                            cutânea
9                                                           epidural
10                                            repiratória (inalação)
11                                                         oftálmica
12                                                             nasal
13                                                            dental
14                                

In [7]:

silver["VIA_ADMINISTRACAO_CHAVE"].value_counts()


VIA_ADMINISTRACAO_CHAVE
P               755
desconhecido    516
O               253
TD              139
Inhal            96
Instill          50
N                21
SL               16
V                11
R                10
Implant           9
Name: count, dtype: int64

In [ ]:
silver.query("VIA_ADMINISTRACAO_CHAVE=='desconhecido'")["VIA_ADMINISTRACAO"].value_counts()

VIA_ADMINISTRACAO
Transplacental                                                 7
tranplacental                                                  7
transmamária                                                   4
E2B R2 Code: 065 - Unknown                                     2
Transplacentária                                               2
E2B R2 Code: 050 - Other                                       2
desconhecida                                                   2
Unknown                                                        2
transplacentária                                               2
outra                                                          2
Endenosa                                                       1
intra- vitrea                                                  1
IVAD                                                           1
MSE                                                            1
EV,,                                                           1
mse    

In [14]:

silver["VIA_ADMINISTRACAO_MAE_PAI_CHAVE"].value_counts()

VIA_ADMINISTRACAO_MAE_PAI_CHAVE
desconhecido    1849
P                 16
O                  9
Implant            1
Inhal              1
Name: count, dtype: int64

In [7]:
silver.head(2)

,DETENTOR_REGISTRO,DETENTOR_REGISTRO_PADRONIZADO
0,ABACUS MEDICINE,ABACUS MEDICINE
1,ABBOLT,ABBOT


In [8]:
silver.DETENTOR_REGISTRO.nunique()

6675

In [9]:
silver.DETENTOR_REGISTRO_PADRONIZADO.nunique()

271

In [10]:
silver.DETENTOR_REGISTRO_PADRONIZADO.value_counts().head(20)

DETENTOR_REGISTRO_PADRONIZADO
DESCONHECIDO              2080
ASTRAZENECA                284
PFIZER                     257
FIOCRUZ                    195
CRISTALIA                  176
JOHNSON & JOHNSON          161
BLAU FARMACÊUTICA          109
EMS                        103
UNIAO QUIMICA               97
MEDICAMENTO MANIPULADO      97
SINOVAC BIONTECH            94
SANOFI                      93
EUROFARMA                   87
ABL                         82
TEUTO                       79
HIPOLABOR                   76
HALEX ISTAR                 68
MERCK                       67
FRESENIUS KABI              60
INSTITUTO BUTANTAN          59
Name: count, dtype: int64

In [11]:
dim = silver[['DETENTOR_REGISTRO']].drop_duplicates().fillna('DESCONHECIDO')
dim.head(20)

,DETENTOR_REGISTRO
0,ABACUS MEDICINE
1,ABBOLT
2,ABBOT
3,ABBOT BRASIL
4,ABBOT L: 0071/20 V: 31/01/2022
5,ABBOT L: 1040818 V: 31/03/2022
6,ABBOT LABORATORIUOS DO BRASIL LTDA
7,ABBOTH BIOLOGICALS
8,ABBOTT
9,ABBOTT B.V.


In [12]:
# termos genéricos que não diferenciam empresas farmacêuticas
STOPWORDS_EMPRESA = {
    "SA", "S/A", "S.A", "S.A.", "SA.", "S A",
    "LTDA", "LTD", "EIRELI", "ME", "EPP",
    "COMERCIO", "COMERCIAL", "INDUSTRIA", "INDUSTRIAL",
    "LABORATORIO", "LABORATORIOS", "LAB",
    "FARMACEUTICA", "FARMACEUTICO", "FARMACEUTICOS","PHARMA",
    "DO", "DA", "DE", "DOS", "DAS", "E",
}

def limpar_texto_empresa(texto: str) -> str:
    """Normaliza texto de nome de empresa para criar uma chave estável."""
    if pd.isna(texto):
        return ""
    
    s = str(texto).strip()
    s = unidecode(s)           # remove acentos
    s = s.upper()              # caixa alta
    s = re.sub(r"[^A-Z0-9 ]+", " ", s)  # tira pontuação
    tokens = [t for t in s.split() if t not in STOPWORDS_EMPRESA]
    s = " ".join(tokens)
    s = re.sub(r"\s+", " ", s).strip()  # colapsa espaços
    
    return s

def criar_key_detentor(texto: str) -> str:
    """Wrapper pra ficar semântico."""
    return limpar_texto_empresa(texto)


In [13]:
dim["DETENTOR_REGISTRO_VALOR"] = dim["DETENTOR_REGISTRO"].apply(criar_key_detentor)
dim["DETENTOR_REGISTRO_VALOR"].nunique()


5404

In [14]:
silver[['DETENTOR_REGISTRO_PADRONIZADO']].nunique()

DETENTOR_REGISTRO_PADRONIZADO    271
dtype: int64

In [15]:
dim = silver[['DETENTOR_REGISTRO_PADRONIZADO']].drop_duplicates().fillna('DESCONHECIDO')
dim["DETENTOR_REGISTRO_VALOR"] = dim["DETENTOR_REGISTRO_PADRONIZADO"].apply(criar_key_detentor)
dim["DETENTOR_REGISTRO_VALOR"].nunique()


265

In [16]:
dim.head()

,DETENTOR_REGISTRO_PADRONIZADO,DETENTOR_REGISTRO_VALOR
0,ABACUS MEDICINE,ABACUS MEDICINE
1,ABBOT,ABBOT
30,ABBVIE,ABBVIE
46,ABL,ABL
128,ACCORD FARMACÊUTICA,ACCORD


In [17]:
freq = (
    dim
    .groupby(["DETENTOR_REGISTRO_VALOR", "DETENTOR_REGISTRO_PADRONIZADO"])
    .size()
    .reset_index(name="n")
)
freq.sort_values("n", ascending=False).head(10)

,DETENTOR_REGISTRO_VALOR,DETENTOR_REGISTRO_PADRONIZADO,n
0,ABACUS MEDICINE,ABACUS MEDICINE,1
1,ABBOT,ABBOT,1
2,ABBVIE,ABBVIE,1
3,ABL,ABL,1
4,ACCORD,ACCORD FARMACÊUTICA,1
5,ACHE,ACHÉ,1
6,ADC THERAPEUTICS,ADC THERAPEUTICS,1
7,ADIUM,ADIUM,1
8,AIRELA,AIRELA,1
9,ALCON,ALCON,1


In [18]:
dim_detentor = (
    freq
    .sort_values(["DETENTOR_REGISTRO_VALOR", "n"], ascending=[True, False])
    .drop_duplicates("DETENTOR_REGISTRO_VALOR")
    #.rename(columns={"DETENTOR_REGISTRO_PADRONIZADO": "DETENTOR_REGISTRO_CANONICO"})
    [["DETENTOR_REGISTRO_VALOR", "DETENTOR_REGISTRO_PADRONIZADO"]]
    .reset_index(drop=True)
)
dim_detentor.head()

,DETENTOR_REGISTRO_VALOR,DETENTOR_REGISTRO_PADRONIZADO
0,ABACUS MEDICINE,ABACUS MEDICINE
1,ABBOT,ABBOT
2,ABBVIE,ABBVIE
3,ABL,ABL
4,ACCORD,ACCORD FARMACÊUTICA


In [19]:
dim_detentor.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 265 entries, 0 to 264
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   DETENTOR_REGISTRO_VALOR        265 non-null    object
 1   DETENTOR_REGISTRO_PADRONIZADO  265 non-null    object
dtypes: object(2)
memory usage: 4.3+ KB


In [20]:
silver[['DETENTOR_REGISTRO_PADRONIZADO']].nunique()

DETENTOR_REGISTRO_PADRONIZADO    271
dtype: int64

In [21]:
dim_detentor =  silver[['DETENTOR_REGISTRO_PADRONIZADO']].drop_duplicates().fillna('DESCONHECIDO')
dim_detentor["DETENTOR_REGISTRO_VALOR"] = dim_detentor["DETENTOR_REGISTRO_PADRONIZADO"].apply(criar_key_detentor)
dim_detentor.insert(0, "DETENTOR_REGISTRO_CHAVE", range(1, len(dim_detentor) + 1))
dim_detentor.info()


<class 'pandas.core.frame.DataFrame'>
Index: 271 entries, 0 to 6764
Data columns (total 3 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   DETENTOR_REGISTRO_CHAVE        271 non-null    int64 
 1   DETENTOR_REGISTRO_PADRONIZADO  271 non-null    object
 2   DETENTOR_REGISTRO_VALOR        271 non-null    object
dtypes: int64(1), object(2)
memory usage: 8.5+ KB


In [22]:
dim_detentor.DETENTOR_REGISTRO_CHAVE.nunique()

271

In [23]:
hist = silver.merge(dim_detentor, on="DETENTOR_REGISTRO_PADRONIZADO", how="left")
hist.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6765 entries, 0 to 6764
Data columns (total 4 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   DETENTOR_REGISTRO              6761 non-null   object
 1   DETENTOR_REGISTRO_PADRONIZADO  6765 non-null   object
 2   DETENTOR_REGISTRO_CHAVE        6765 non-null   int64 
 3   DETENTOR_REGISTRO_VALOR        6765 non-null   object
dtypes: int64(1), object(3)
memory usage: 211.5+ KB


In [24]:
hist.head()

,DETENTOR_REGISTRO,DETENTOR_REGISTRO_PADRONIZADO,DETENTOR_REGISTRO_CHAVE,DETENTOR_REGISTRO_VALOR
0,ABACUS MEDICINE,ABACUS MEDICINE,1,ABACUS MEDICINE
1,ABBOLT,ABBOT,2,ABBOT
2,ABBOT,ABBOT,2,ABBOT
3,ABBOT BRASIL,ABBOT,2,ABBOT
4,ABBOT L: 0071/20 V: 31/01/2022,ABBOT,2,ABBOT


In [25]:
dim_detentor.DETENTOR_REGISTRO_PADRONIZADO.nunique()

271

In [26]:
dim_detentor.DETENTOR_REGISTRO_CHAVE.nunique()

271

In [27]:
hist.to_parquet(Path(project_root) / "data/02_silver/hist_dim_detentor_registro/hist_dim_detentor_registro.parquet", index=False)

# 🥇 Gold


In [28]:
hist.columns

Index(['DETENTOR_REGISTRO', 'DETENTOR_REGISTRO_PADRONIZADO',
       'DETENTOR_REGISTRO_CHAVE', 'DETENTOR_REGISTRO_VALOR'],
      dtype='object')

In [29]:
dim = hist[['DETENTOR_REGISTRO_CHAVE', 'DETENTOR_REGISTRO_VALOR']].drop_duplicates()
dim.to_parquet(Path(project_root) / "data/03_gold/dim_detentor_registro/dim_detentor_registro.parquet", index=False)

In [31]:

dim.to_csv(Path(project_root) / "data/03_gold/dim_detentor_registro/dim_detentor_registro.csv", sep=";", index=False)